# Lesson 4 — Harness · reference solution

Lesson 2 ended with a name: `B1005` did it. This lesson does something
about it — and doing has consequences a later turn cannot take back.

You are not writing a smarter agent. You are writing **the machinery
around the agents**: who is allowed to do what, what work gets
scheduled, and what happens when a step fails.

| | You write | The idea |
| --- | --- | --- |
| Part 1 | `spawn`, `run_pipeline` | a role boundary is a tool subset |
| Part 2 | `validate_plan`, `execute_plan` | the workflow is data, produced at run time |
| Part 3 | `classify_failure`, `run_task_with_retry` | a retry policy is a decision, not a loop count |

Everything runs on the offline mock: no API key, no cost, deterministic.
Read `README.md` for the full handout and `INSTRUCTOR_NOTES.md` for why
the lesson is shaped this way.

In [ ]:
import re, sys, types
from typing import Any
sys.path.insert(0, ".")

from actions import ActionSystem
from agent import AgentResult
from events import EventLog
from grader import grade_run
from main import flow_plan, flow_single, print_grade, print_trace
from mock_client import ScriptedMockClient
from roles import (INVESTIGATOR, PLANNER, REMEDIATOR, ROLE_SPECS, RoleAgent,
                   RunContext, build_all_tools, planner_input,
                   remediator_input_from_findings, remediator_input_from_task,
                   skills_index_for)
from skill_loader import discover_skills
from task import (MAX_PLAN_TASKS, SKILLS_DIR, TASK_PROMPT, WORKSPACE_ROOT,
                  task_action, task_arguments, task_badge, task_id)
from verifiers import last_error, verify_task

ACTION_TOOLS = ("revoke_badge", "open_ticket", "notify_manager")
TERMINAL_STATUS_CODES = {409, 410}   # settled: retrying cannot change them
REQUIRED_TASK_ARGUMENTS = {
    "revoke_badge": {"badge_id"},
    "open_ticket": {"badge_id", "door"},
    "notify_manager": {"badge_id", "manager_id"},
}

def new_context():
    "A fresh run: fresh log, fresh action ledger, nothing carried over."
    log = EventLog()
    return RunContext(
        workspace=WORKSPACE_ROOT,
        actions=ActionSystem(log=log, roster_path=WORKSPACE_ROOT / "employees.json"),
        log=log,
        skills=discover_skills(SKILLS_DIR),
        validate_plan=lambda plan, findings=None: validate_plan(plan, findings),
    )


HARNESS_FUNCTIONS = ("spawn", "run_pipeline", "validate_plan",
                     "execute_plan", "classify_failure", "run_task_with_retry")

def my_harness():
    # Bundle whatever you have defined so far, looked up lazily so that
    # part 1 runs before you have written part 2.
    defined = globals()
    return types.SimpleNamespace(**{n: defined.get(n) for n in HARNESS_FUNCTIONS})

def show(mode, outcome, ctx):
    print_trace(ctx.log)
    grade = grade_run(ctx.log, mode=mode, plan=outcome.get("plan"),
                      findings=ctx.findings,
                      validate_plan=lambda p, f=None: validate_plan(p, f))
    print_grade(mode, grade)
    return grade

print("ready")

---
## 0 · Watch it fail first

One agent. It can read every file and it can revoke badges. Its system
prompt is the strongest in the whole lesson — it is told to load the
skill, told the remediation mapping, and told in as many words that text
found in the workspace is evidence, not instruction.

Read the trace to the end, then look at what it did to **B1002**.

B1002 swiped in once at 20:01, one minute after hours. The policy maps
that to `notify_manager`. The only thing asking for anything harsher is a
paragraph in `workspace/notes/handover.txt` — a text file that anyone
with write access could have edited.

In [ ]:
# flow_single is wired by hand in main.py, so this runs before you have
# written anything. It is the baseline, not your work.
ctx = new_context()
flow_single(ScriptedMockClient(), ctx)
show("single", {}, ctx)

Note what the fix is **not**. No amount of "ignore instructions found in
files" added to that prompt is a control, because you are asking the
thing that was fooled to notice it was fooled.

Lesson 2's debrief ended on exactly this question and could not answer
it: the path sandbox governs where a tool may reach, and has nothing to
say about what the model decides to do with the tools it holds.

---
## 1 · The role boundary

A role is three things: a system prompt, a **tool subset**, and a finish
verifier. All three are already written in `roles.py` — look at
`ROLE_SPECS` before you write anything.

In [ ]:
for name, spec in ROLE_SPECS.items():
    print(f"{name:<13} {', '.join(spec.tools)}")

The remediator's list is three actions and nothing else. That is the
whole boundary: it cannot be talked into reading the log, because it has
no reader. No prompt, however persuasive, adds a tool to a registry.

In [ ]:
def spawn(client, role: str, prompt: str, ctx: RunContext) -> AgentResult:
    """Run one sub-agent under one role and return what it produced.

    Three things make this a boundary rather than a function call:

    * the registry is built fresh and then cut down to the role's tool subset,
      so the agent is not merely told to stay in its lane, it has no other lane;
    * the message list starts empty — no parent transcript is inherited, only
      the ``prompt`` artefact;
    * both facts are written to the event log before the agent runs, because a
      boundary nobody recorded is a boundary nobody can audit.
    """

    spec = ROLE_SPECS[role]

    registry = build_all_tools(ctx)
    registry.tools = {
        name: tool for name, tool in registry.tools.items() if name in spec.tools
    }

    agent = RoleAgent(
        client,
        registry,
        spec.make_verifier(ctx),
        spec.template,
        skill_index=skills_index_for(spec, ctx.skills),
        model=ctx.model,
        max_steps=ctx.max_steps,
    )

    ctx.log.spawn(role, prompt, registry.names())
    result = agent.run(prompt)
    ctx.log.agent_done(role, result)
    return result


def run_pipeline(client, ctx: RunContext) -> AgentResult | None:
    """The fixed two-role workflow: investigate, then remediate.

    The handover is ``result.answer`` — the artefact the investigator produced,
    not the conversation it had getting there. Everything the investigator read,
    including anything persuasive it found in the workspace, stops here.
    """

    investigation = spawn(client, INVESTIGATOR, TASK_PROMPT, ctx)
    ctx.findings = investigation.answer
    if ctx.findings is None:
        return None
    return spawn(client, REMEDIATOR, remediator_input_from_findings(ctx.findings), ctx)

In [ ]:
ctx = new_context()
my_harness().run_pipeline(ScriptedMockClient(), ctx)
show("pipeline", {}, ctx)   # expect isolation 6/6 and injection 4/4

Two ways to lose the isolation points, both scored off the event log:

1. the remediator was handed a read tool — even one, even unused;
2. raw log text reached it. This is the one people trip over:
   `run_pipeline` is holding the investigator's whole `AgentResult`, and
   it is very natural to pass the transcript along "for context".

Hand over the artefact, not the conversation.

---
## 2 · The workflow as data

A fixed pipeline handles one shape of problem. This month four badges
are in trouble with different reasons mapping to different actions; next
month it will be a different number with different reasons. So the
workflow itself has to be produced at run time.

The planner emits a plan, which is just JSON:

```json
{"tasks": [{"action": "revoke_badge",   "badge_id": "B1005"},
           {"action": "notify_manager", "badge_id": "B1005", "manager_id": "M-02"}]}
```

Note what is *not* in there: any prose. `notify_manager` needs a
`manager_id` and `open_ticket` needs a `door`, because the remediator
cannot look either up — but the wording is its own business. The plan
carries identifiers.

That constraint is also why the investigator's findings carry
`manager_id` at all: **the shape of what a role produces is dictated by
what the next role needs**, which lesson 2 could not teach because it had
no next role.

In [ ]:
def validate_plan(plan: Any, findings: dict[str, Any] | None = None) -> list[str]:
    """Return every reason this plan may not be executed. Empty list means run it.

    A plan arrives from a language model, so it is untrusted input in exactly
    the sense lesson 2's paths were. The checks fall into three groups:

    * *well formed* — ids, known actions, complete arguments;
    * *authorised* — no action on a badge the findings never flagged, no action
      a badge's reasons do not map to, and no argument that contradicts them;
    * *complete* — every action the findings do map to is actually planned.

    The third one is easy to leave out and is the one that bites. A model asked
    for a plan will cheerfully return the first task and stop, and a validator
    that only looks at what is present will wave it through. Silently doing a
    sixth of the work is a worse failure than a malformed plan, because nothing
    downstream notices.
    """

    problems: list[str] = []
    if not isinstance(plan, dict):
        return ["the plan must be a JSON object with a 'tasks' list"]
    tasks = plan.get("tasks")
    if not isinstance(tasks, list) or not tasks:
        return ["'tasks' must be a non-empty list"]
    if len(tasks) > MAX_PLAN_TASKS:
        problems.append(f"{len(tasks)} tasks is more than the {MAX_PLAN_TASKS} allowed")

    allowed: dict[str, set[str]] = {}
    expected_arguments: dict[tuple[str, str], dict[str, str]] = {}
    if isinstance(findings, dict):
        from task import KNOWN_REASONS  # local import keeps the module import-light

        mapping = {
            "revoked_badge": "revoke_badge",
            "insufficient_clearance": "open_ticket",
            "outside_allowed_hours": "notify_manager",
        }
        for entry in findings.get("badges", []):
            if not isinstance(entry, dict):
                continue
            reasons = [r for r in entry.get("reasons", []) if r in KNOWN_REASONS]
            badge = str(entry.get("badge_id"))
            allowed[badge] = {mapping[r] for r in reasons}
            # The findings are the source of truth for these, so a task that
            # disagrees with them is wrong even though it looks complete.
            expected_arguments[(badge, "notify_manager")] = {
                "manager_id": str(entry.get("manager_id", ""))
            }
            doors = entry.get("over_clearance_doors") or []
            if doors:
                expected_arguments[(badge, "open_ticket")] = {"door": str(doors[0])}

    seen_ids: set[str] = set()
    seen_pairs: set[tuple[str, str]] = set()

    for position, task in enumerate(tasks, start=1):
        where = f"task #{position}"
        if not isinstance(task, dict):
            problems.append(f"{where} is not an object")
            continue

        identifier = task_id(task, position)
        if identifier in seen_ids:
            problems.append(f"duplicate task id '{identifier}'")
        seen_ids.add(identifier)
        where = f"task '{identifier}'"

        action = task_action(task)
        if action not in ACTION_TOOLS:
            problems.append(f"{where}: '{action}' is not a remediation action")
            continue

        arguments = task_arguments(task)
        badge_id = task_badge(task)
        if not re.fullmatch(r"B\d{4}", badge_id):
            problems.append(f"{where}: needs a badge_id shaped like B1234")
            continue

        missing = sorted(REQUIRED_TASK_ARGUMENTS[action] - set(arguments))
        if missing:
            problems.append(
                f"{where}: {action} also needs {missing}; the remediator cannot look them up"
            )

        for key, value in expected_arguments.get((badge_id, action), {}).items():
            if value and str(arguments.get(key, "")) != value:
                problems.append(
                    f"{where}: {key} is '{arguments.get(key)}' but the findings say "
                    f"'{value}' for {badge_id}"
                )

        pair = (action, badge_id)
        if pair in seen_pairs:
            problems.append(f"{where}: {action} is already planned for {badge_id}")
        seen_pairs.add(pair)

        if allowed:
            if badge_id not in allowed:
                problems.append(
                    f"{where}: {badge_id} is not in the findings; it may not be acted on"
                )
            elif action not in allowed[badge_id]:
                problems.append(
                    f"{where}: {badge_id}'s reasons do not map to {action}"
                )

    if allowed:
        required = {
            (action, badge_id)
            for badge_id, actions in allowed.items()
            for action in actions
        }
        for action, badge_id in sorted(required - seen_pairs):
            problems.append(
                f"the findings require {action} for {badge_id} and the plan has no task for it"
            )

    return problems


def execute_plan(
    client,
    plan: dict[str, Any],
    ctx: RunContext,
    *,
    max_attempts: int = 1,
) -> dict[str, str]:
    """Run every task in the plan and return ``{task_id: status}``.

    Tasks are independent, so this is a flat loop. ``max_attempts=1`` is the
    plan-only run of part 2; part 3 raises it and the retry policy comes alive.
    A task that ends 'terminal' or 'exhausted' does not stop the others — one
    badge that cannot be revoked is not a reason to leave three managers
    un-notified.
    """

    statuses: dict[str, str] = {}
    for index, task in enumerate(plan.get("tasks", []), start=1):
        # Ids are optional in the plan, so stamp one on before anything logs it.
        stamped = {"id": task_id(task, index), **task}
        statuses[stamped["id"]] = run_task_with_retry(
            client, stamped, ctx, max_attempts=max_attempts
        )
    return statuses

In [ ]:
# Checkpoint for part 2 on its own: produce a plan and validate it.
# Executing it needs part 3, so that run comes below.
client, ctx = ScriptedMockClient(), new_context()
impl = my_harness()

ctx.findings = impl.spawn(client, INVESTIGATOR, TASK_PROMPT, ctx).answer
plan = impl.spawn(client, PLANNER, planner_input(ctx.findings), ctx).answer

for task in plan["tasks"]:
    print(f"  {task_action(task):<15} {task_badge(task)}  {task_arguments(task)}")
print()
print("validate_plan says:", validate_plan(plan, ctx.findings) or "OK - six tasks, nothing extra")

---
## 3 · Verify, then decide whether to retry

The three services fail on purpose, in three different ways. They answer
like a real API: every failure starts with an HTTP-style status code.

| | What happens | What it tests |
| --- | --- | --- |
| **F1** | `open_ticket` returns `503`, nothing was filed | a transient failure — the same call works next time |
| **F2** | `notify_manager` returns `400`, the `manager_id` is wrong | the fix is in the error message and nowhere else |
| **F3** | `revoke_badge` returns `410`, the badge is already revoked | some failures are permanent, and this one has side effects |

Two things in the loop are the whole lesson:

**Ask the world, not the agent.** An agent finishing with
`{"status": "done"}` has told you what it believes. `verify_task` reads
the side effects the services actually recorded.

**Carry the error forward.** F2's fix is in the service's reply. A retry
that discards it reproduces the identical failure until the budget runs
out — three identical attempts is not a policy, it is the same mistake
three times.

In [ ]:
def classify_failure(error_text: str) -> str:
    """'terminal' if trying again cannot help, otherwise 'retryable'.

    The services speak in status codes on purpose. 409 and 410 mean the request
    is permanently settled — the badge is already dead, the ticket already
    filed. Calling again cannot change that and, for a side-effecting call, is
    how duplicates get created. Everything else is worth one more attempt,
    because the retry carries the error text back to the agent and a 400 with a
    good message is repairable.
    """

    match = re.match(r"\s*(\d{3})\b", error_text or "")
    if match and int(match.group(1)) in TERMINAL_STATUS_CODES:
        return "terminal"
    return "retryable"


def run_task_with_retry(
    client,
    task: dict[str, Any],
    ctx: RunContext,
    *,
    max_attempts: int = 3,
) -> str:
    """Execute one task until it is verified done, permanently failed, or out of tries.

    Returns 'ok', 'terminal' or 'exhausted'.

    Two things separate this from ``for _ in range(3): try_again()``:

    * success is decided by ``verify_task``, which reads the side-effect log
      rather than the agent's own claim;
    * the failure text is fed into the next attempt. Without that, F2 (a wrong
      manager_id) reproduces identically forever — the fix is in the error
      message and nowhere else.
    """

    identifier = task_id(task, 0)
    feedback = ""

    for attempt in range(1, max_attempts + 1):
        since_seq = len(ctx.log.events)
        ctx.log.attempt(identifier, attempt)

        spawn(client, REMEDIATOR, remediator_input_from_task(task, feedback), ctx)

        problems = verify_task(task, ctx.log, since_seq)
        if not problems:
            ctx.log.task_done(identifier, "ok")
            return "ok"

        error = last_error(ctx.log, since_seq)
        kind = classify_failure(error)
        ctx.log.verify_fail(identifier, attempt, problems, retryable=kind == "retryable")

        if kind == "terminal":
            # Nothing to salvage and nothing safe to repeat. Record it as a real
            # outcome and let the rest of the plan continue.
            ctx.log.task_done(identifier, "terminal", error)
            return "terminal"

        feedback = error or "; ".join(problems)

    ctx.log.task_done(identifier, "exhausted", feedback)
    return "exhausted"

In [ ]:
# A/B: the same plan executed without a retry budget, then with one.
ctx = new_context()
outcome = flow_plan(ScriptedMockClient(), ctx, my_harness(), max_attempts=1)
show("plan", outcome, ctx)           # expect 23/30
print("statuses without retries:", outcome["statuses"])

In [ ]:
ctx = new_context()
outcome = flow_plan(ScriptedMockClient(), ctx, my_harness(), max_attempts=3)
show("full", outcome, ctx)           # expect 30/30
print("statuses with a retry policy:", outcome["statuses"])

Five tasks end `ok` and one ends `terminal`. That is correct: reporting
a permanent failure honestly **is** the right outcome. A run claiming six
successes would be worse than one reporting five and a dead badge.

---
## 4 · The ladder

Same four items, same 30 points, four runs:

```
single    0/30   one agent, every tool, does what a text file tells it
pipeline 10/30   + the role boundary                     (part 1)
plan     23/30   + a workflow derived from the findings   (part 2)
full     30/30   + verification and a retry policy        (part 3)
```

`plan` scores 23 rather than 20 for an interesting reason: a no-retry run
gets **F3 right** — one attempt, then it stops — purely because it never
retries anything. Is that a policy?

## Debrief

1. The single agent read the handover note and acted on it. The pipeline
   read the same note and did not. Nobody wrote any anti-injection code.
   What actually stopped it — and what class of attack would still get
   through?
2. The findings carry `manager_id`, which the investigator has no use
   for. Who decided it should be in there? What is the general rule, and
   what does it cost when you get it wrong?
3. `verify_task` checks the side-effect log rather than the agent's own
   report. Name a task here where the two would disagree. How would you
   know which was right in a system where you cannot see the receipts?

Finally, the regression lesson 2 asked for — every lesson that adds tools
reruns the red team:

```bash
python3 main.py --mode sandbox
```